## **Decision Tree Hyperparameters and Feature Importance**

### **Topic Roadmap**

**1. Prepare a classification dataset**

**2. Compare tree complexity**

**3. Tune hyperparameters with cross-validation**

**4. Interpret impurity-based and permutation importance**

**5. Key revision notes**

## **1. Prepare the Dataset**

A reproducible binary classification problem is used to isolate the effect of tree hyperparameters.

In [1]:
import pandas as pd
RANDOM_STATE = 42
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score

X, y = make_classification(
    n_samples=1200, n_features=8, n_informative=4, n_redundant=2,
    class_sep=1.2, random_state=RANDOM_STATE
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

## **2. Compare Tree Complexity**

A deeper tree can reduce training error but may increase test error. Comparing depth makes the bias-variance trade-off explicit.

In [2]:
RANDOM_STATE = 42
depth_results = []
for depth in [1, 2, 3, 5, 8, None]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    depth_results.append({
        "max_depth": depth,
        "train_accuracy": model.score(X_train, y_train),
        "test_accuracy": model.score(X_test, y_test),
    })
pd.DataFrame(depth_results)

,max_depth,train_accuracy,test_accuracy
0,1.0,0.683333,0.662500
1,2.0,0.703125,0.675000
2,3.0,0.838542,0.787500
3,5.0,0.912500,0.866667
4,8.0,0.970833,0.887500
5,NaN,1.000000,0.904167


## **3. Tune Hyperparameters**

`GridSearchCV` evaluates a small, interpretable search space using cross-validation. The selected model is refit on the complete training split.

In [3]:
param_grid = {
    "criterion": ["gini", "entropy", "log_loss"],
    "max_depth": [2, 3, 5, 8, None],
    "min_samples_leaf": [1, 5, 10],
    "ccp_alpha": [0.0, 0.001, 0.01],
}
search = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid=param_grid, cv=5, scoring="accuracy", n_jobs=-1
)
search.fit(X_train, y_train)
print(search.best_params_)
print(f"Best CV accuracy: {search.best_score_:.3f}")

{'ccp_alpha': 0.0, 'criterion': 'entropy', 'max_depth': None, 'min_samples_leaf': 1}
Best CV accuracy: 0.899


In [4]:
best_tree = search.best_estimator_
y_pred = best_tree.predict(X_test)
print(f"Test accuracy: {accuracy_score(y_test, y_pred):.3f}")

Test accuracy: 0.925


## **4. Feature Importance**

Impurity-based importance is fast but can favor high-cardinality or noisy features. Permutation importance measures the score decrease after shuffling one feature on unseen data.

In [5]:
feature_names = [f"feature_{i}" for i in range(X.shape[1])]
imp_df = pd.DataFrame({
    "feature": feature_names,
    "impurity_importance": best_tree.feature_importances_,
})
perm = permutation_importance(best_tree, X_test, y_test, n_repeats=20, random_state=RANDOM_STATE, n_jobs=-1)
imp_df["permutation_mean"] = perm.importances_mean
imp_df.sort_values("permutation_mean", ascending=False)

,feature,impurity_importance,permutation_mean
5,feature_5,0.264724,0.232292
1,feature_1,0.230637,0.189375
4,feature_4,0.135678,0.170000
6,feature_6,0.147392,0.169583
3,feature_3,0.080702,0.096250
0,feature_0,0.101499,0.060833
2,feature_2,0.020226,0.003958
7,feature_7,0.019142,0.001667


### **Key Revision Notes**

- `max_depth` limits the longest rule path.
- `min_samples_split` and `min_samples_leaf` prevent very small terminal nodes.
- `ccp_alpha` applies cost-complexity pruning.
- Impurity importance is model-internal; permutation importance is evaluated on a chosen validation set.